In [1]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "anndata>=0.12.11",
#     "scanpy>=1.12.1",
#     "zarr>=3.1.6",
# ]
# ///

In [2]:
from pathlib import Path

In [3]:
import anndata as ad
import numpy as np
import scanpy as sc

In [4]:
INPUT_PATH = Path("habib17.h5ad")
RAW_OUTPUT_PATH = Path("test-data/habib17.zarr")
OUTPUT_PATH = Path("test-data/habib17-differential-expression-test-data.zarr")
GROUPBY_COLUMN = "CellType"

In [5]:
def _sample_expression_values(x: object, max_items: int = 10000) -> np.ndarray:
    if hasattr(x, "tocoo"):
        values = np.asarray(x.data)
    else:
        values = np.asarray(x).ravel()

    if values.size == 0:
        return values

    if values.size > max_items:
        step = max(1, values.size // max_items)
        values = values[::step]

    return values

In [6]:
def _needs_preprocessing(adata: ad.AnnData) -> tuple[bool, str]:
    if "log1p" in adata.uns:
        return False, "Detected adata.uns['log1p']; matrix appears already log-transformed."

    values = _sample_expression_values(adata.X)
    if values.size == 0:
        return True, "Empty matrix sample; applying preprocessing by default."

    tol = 1e-6
    non_integer_fraction = float(np.mean(np.abs(values - np.round(values)) > tol))
    max_value = float(np.max(values))

    # Heuristic: mostly non-integers with compressed range usually indicates logged data.
    if non_integer_fraction > 0.2 and max_value < 50:
        return (
            False,
            (
                "Expression values look already transformed "
                f"(non-integer fraction={non_integer_fraction:.3f}, max={max_value:.3f})."
            ),
        )

    return (
        True,
        (
            "Expression values look like raw counts "
            f"(non-integer fraction={non_integer_fraction:.3f}, max={max_value:.3f})."
        ),
    )

In [7]:
def main() -> None:
    if not INPUT_PATH.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_PATH}")

    adata = ad.read_h5ad(INPUT_PATH)

    if GROUPBY_COLUMN not in adata.obs.columns:
        available = ", ".join(adata.obs.columns.astype(str).tolist())
        raise KeyError(
            f"Column '{GROUPBY_COLUMN}' not found in obs. Available columns: {available}"
        )

    # Align with the zarr v3 output style used in other dataset creation scripts.
    ad.settings.zarr_write_format = 3
    ad.settings.write_csr_csc_indices_with_min_possible_dtype = True
    ad.settings.auto_shard_zarr_v3 = True

    RAW_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    adata.write_zarr(RAW_OUTPUT_PATH)

    print(f"Wrote source AnnData to {RAW_OUTPUT_PATH}")

    # Make sure group labels are categorical for rank_genes_groups.
    adata.obs[GROUPBY_COLUMN] = adata.obs[GROUPBY_COLUMN].astype("category")

    should_preprocess, reason = _needs_preprocessing(adata)
    print(reason)

    if should_preprocess:
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        print("Applied preprocessing: normalize_total + log1p")
    else:
        print("Skipped preprocessing")

    sc.tl.rank_genes_groups(
        adata,
        groupby=GROUPBY_COLUMN,
        method="wilcoxon",
        use_raw=False,
    )

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    adata.write_zarr(OUTPUT_PATH)

    print(f"Wrote DE results to {OUTPUT_PATH}")

In [8]:
if __name__ == "__main__":
    main()

/home/klaus/.cache/uv/environments-v2/juv-tmp-hndd5-7u-13d136009a0f9047/lib/python3.12/site-packages/zarr/core/array.py:4442: ZarrUserWarning: Automatic shard shape inference is experimental and may change without notice.
  shard_shape_parsed, chunk_shape_parsed = _auto_partition(
/home/klaus/.cache/uv/environments-v2/juv-tmp-hndd5-7u-13d136009a0f9047/lib/python3.12/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Wrote source AnnData to test-data/habib17.zarr
Expression values look already transformed (non-integer fraction=1.000, max=5.169).
Skipped preprocessing


/home/klaus/.cache/uv/environments-v2/juv-tmp-hndd5-7u-13d136009a0f9047/lib/python3.12/site-packages/zarr/core/array.py:4442: ZarrUserWarning: Automatic shard shape inference is experimental and may change without notice.
  shard_shape_parsed, chunk_shape_parsed = _auto_partition(


Wrote DE results to test-data/habib17-differential-expression-test-data.zarr


/home/klaus/.cache/uv/environments-v2/juv-tmp-hndd5-7u-13d136009a0f9047/lib/python3.12/site-packages/zarr/core/dtype/npy/structured.py:591: UnstableSpecificationWarning: The data type (Struct(fields=(('ASC1', FixedLengthUTF32(length=17, endianness='little')), ('ASC2', FixedLengthUTF32(length=17, endianness='little')), ('END', FixedLengthUTF32(length=17, endianness='little')), ('GABA1', FixedLengthUTF32(length=17, endianness='little')), ('GABA2', FixedLengthUTF32(length=17, endianness='little')), ('MG', FixedLengthUTF32(length=17, endianness='little')), ('NSC', FixedLengthUTF32(length=17, endianness='little')), ('ODC1', FixedLengthUTF32(length=17, endianness='little')), ('OPC', FixedLengthUTF32(length=17, endianness='little')), ('Unclassified', FixedLengthUTF32(length=17, endianness='little')), ('exCA1', FixedLengthUTF32(length=17, endianness='little')), ('exCA3', FixedLengthUTF32(length=17, endianness='little')), ('exDG', FixedLengthUTF32(length=17, endianness='little')), ('exPFC1', Fi